# CorePromoter TSS/PAS pretraining

This notebook creates a species-grouped TSS/PAS dataset, pretrains architecture recognition, recalibrates expression on the existing small libraries, and saves a frozen checkpoint for recursive design. It never overwrites `weights_CorePromoter_clean.pt`.

In [1]:
# === Cell 1: setup and editable training config ===
import importlib
from datetime import datetime

import pandas as pd
import torch

import recursive_corepromoter_design as legacy
import tss_pas_dataset as tss_data
import train_corepromoter_tss_pas as trainer
import evaluate_corepromoter_tss_pas as evaluator

importlib.reload(legacy)
importlib.reload(tss_data)
importlib.reload(trainer)
importlib.reload(evaluator)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATASET_DIR = legacy.PROJECT_ROOT / "outputs" / "tss_pas_dataset"
DATASET_PATH = DATASET_DIR / "tss_pas_processed.pkl"
RUN_DIR = legacy.PROJECT_ROOT / "outputs" / "corepromoter_tss_pas" / RUN_STAMP
BASELINE_CHECKPOINT = legacy.WEIGHTS_DIR / "weights_CorePromoter_clean.pt"
ARCH_CHECKPOINT = legacy.WEIGHTS_DIR / "weights_CorePromoter_tss_arch.pt"
FINAL_CHECKPOINT = legacy.WEIGHTS_DIR / "weights_CorePromoter_tss_pas.pt"

TRAINING_CONFIG = trainer.TrainingConfig(
    batch_size=512,
    arch_epochs=20,
    head_epochs=15,
    joint_epochs=15,
)

RUN_STRICT_LOLO = False  # Expensive: retrains both models for all seven held-out libraries.
print("Device:", DEVICE)
print("Run directory:", RUN_DIR)
print("Final checkpoint:", FINAL_CHECKPOINT)

Device: cuda
Run directory: c:\Users\User\OneDrive\桌面\Claude code\promoter library exp\outputs\corepromoter_tss_pas\20260721_020756
Final checkpoint: c:\Users\User\OneDrive\桌面\Claude code\promoter library exp\MS2_Data_PyTorch\weights\weights_CorePromoter_tss_pas.pt


In [2]:
# === Cell 2: prepare and verify the TSS/PAS dataset ===
dataset_outputs = tss_data.save_processed_dataset(
    xlsx_path=tss_data.DEFAULT_XLSX,
    output_dir=DATASET_DIR,
    validation_fraction=0.15,
    test_fraction=0.15,
    seed=legacy.SEED,
    allow_n=True,
)
display(pd.read_csv(dataset_outputs["qc_summary"]))
display(pd.read_csv(dataset_outputs["qc_failure_summary"]).head(20))
display(pd.read_csv(dataset_outputs["split_summary"]))
display(pd.read_csv(dataset_outputs["split_groups"]))
print("Processed dataset:", dataset_outputs["dataset"])

,metric,value
0,total_rows,75935
1,qc_pass_rows,72613
2,qc_fail_rows,3322
3,spacer_length_15,220
4,spacer_length_16,14738
5,spacer_length_17,49157
6,spacer_length_18,11443
7,spacer_length_19,377


,qc_reason,rows
0,Dis_invalid_*,2721
1,unsupported_spacer_19,368
2,unsupported_spacer_15,207
3,unsupported_spacer_15|Dis_invalid_*,13
4,unsupported_spacer_19|Dis_invalid_*,9
5,Start_length_0|ITR_length_0,4


,split,spacer_length,rows
0,test,16,1218
1,test,17,4814
2,test,18,1099
3,train,16,11251
4,train,17,36628
5,train,18,8617
6,validation,16,1615
7,validation,17,5939
8,validation,18,1432


,species_group,split
0,Ca.je,test
1,Ch.tr,test
2,Cl.di,test
3,Ge.su,test
4,La.ph,test
5,Ne.go,test
6,Pa.ri,test
7,Ph.sp,test
8,Ac.ba,train
9,An.sp,train


Processed dataset: c:\Users\User\OneDrive\桌面\Claude code\promoter library exp\outputs\tss_pas_dataset\tss_pas_processed.pkl


In [3]:
# === Cell 3: train once and save frozen checkpoints ===
training_outputs = trainer.train_production_model(
    dataset_path=DATASET_PATH,
    baseline_checkpoint=BASELINE_CHECKPOINT,
    arch_checkpoint=ARCH_CHECKPOINT,
    final_checkpoint=FINAL_CHECKPOINT,
    run_dir=RUN_DIR,
    device=DEVICE,
    config=TRAINING_CONFIG,
)
display(pd.read_csv(training_outputs["metrics"]))
print("Architecture checkpoint:", training_outputs["arch_checkpoint"])
print("Final recursive checkpoint:", training_outputs["final_checkpoint"])

Architecture [01/20] loss=1.1563 top1=0.6968 margin=0.4839
Architecture [02/20] loss=1.1046 top1=0.7152 margin=0.5375
Architecture [03/20] loss=1.0589 top1=0.7308 margin=0.5858
Architecture [04/20] loss=1.0139 top1=0.7438 margin=0.6298
Architecture [05/20] loss=0.9736 top1=0.7553 margin=0.6697
Architecture [06/20] loss=0.9388 top1=0.7649 margin=0.7063
Architecture [07/20] loss=0.9065 top1=0.7741 margin=0.7401
Architecture [08/20] loss=0.8777 top1=0.7849 margin=0.7713
Architecture [09/20] loss=0.8504 top1=0.7933 margin=0.8003
Architecture [10/20] loss=0.8259 top1=0.7997 margin=0.8276
Architecture [11/20] loss=0.8027 top1=0.8081 margin=0.8534
Architecture [12/20] loss=0.7823 top1=0.8140 margin=0.8782
Architecture [13/20] loss=0.7626 top1=0.8204 margin=0.9019
Architecture [14/20] loss=0.7448 top1=0.8262 margin=0.9246
Architecture [15/20] loss=0.7276 top1=0.8293 margin=0.9465
Architecture [16/20] loss=0.7114 top1=0.8323 margin=0.9678
Architecture [17/20] loss=0.6952 top1=0.8346 margin=0.98

,evaluation,metric,value
0,baseline_architecture_validation,n,8986.000000
1,baseline_architecture_validation,top1_architecture_accuracy,0.675161
2,baseline_architecture_validation,m35_position_accuracy,0.701313
3,baseline_architecture_validation,m10_position_accuracy,0.775874
4,baseline_architecture_validation,spacer_accuracy,0.745270
5,baseline_architecture_validation,mean_target_margin,0.424267
6,baseline_architecture_validation,median_target_margin,0.498563
7,baseline_architecture_validation,positive_margin_rate,0.675161
8,baseline_architecture_test,n,7131.000000
9,baseline_architecture_test,top1_architecture_accuracy,0.641986


Architecture checkpoint: c:\Users\User\OneDrive\桌面\Claude code\promoter library exp\MS2_Data_PyTorch\weights\weights_CorePromoter_tss_arch.pt
Final recursive checkpoint: c:\Users\User\OneDrive\桌面\Claude code\promoter library exp\MS2_Data_PyTorch\weights\weights_CorePromoter_tss_pas.pt


In [4]:
# === Cell 4: compare the saved baseline and TSS/PAS checkpoints ===
processed_tss = pd.read_pickle(DATASET_PATH)
expression_data = legacy.build_core_training_dataframe()
comparison = evaluator.checkpoint_comparison(
    BASELINE_CHECKPOINT,
    FINAL_CHECKPOINT,
    processed_tss,
    expression_data,
    DEVICE,
    TRAINING_CONFIG.batch_size,
)
comparison_path = RUN_DIR / "checkpoint_comparison_metrics.csv"
comparison.to_csv(comparison_path, index=False)
display(comparison)
print("Saved:", comparison_path)

,model,evaluation,group,metric,value
0,baseline,species_group_held_out_architecture_test,all_test_species,n,7131.000000
1,baseline,species_group_held_out_architecture_test,all_test_species,top1_architecture_accuracy,0.641986
2,baseline,species_group_held_out_architecture_test,all_test_species,m35_position_accuracy,0.673117
3,baseline,species_group_held_out_architecture_test,all_test_species,m10_position_accuracy,0.715748
4,baseline,species_group_held_out_architecture_test,all_test_species,spacer_accuracy,0.740569
...,...,...,...,...,...
123,tss_pas,checkpoint_per_library_diagnostic,ITS,mae_log10,0.449448
124,tss_pas,checkpoint_per_library_diagnostic,ITS,pearson_r,0.530717
125,tss_pas,checkpoint_per_library_diagnostic,ITS,spearman_rho,0.540019
126,tss_pas,checkpoint_per_library_diagnostic,ITS,r2,-0.145039


Saved: c:\Users\User\OneDrive\桌面\Claude code\promoter library exp\outputs\corepromoter_tss_pas\20260721_020756\checkpoint_comparison_metrics.csv


In [5]:
# === Cell 5: optional strict leave-one-library-out comparison ===
# This is intentionally off by default because it retrains 14 fold models.
if RUN_STRICT_LOLO:
    lolo_metrics, lolo_history = evaluator.leave_one_library_out_comparison(
        expression_data,
        processed_tss,
        DEVICE,
        TRAINING_CONFIG,
        baseline_epochs=50,
    )
    lolo_metrics.to_csv(RUN_DIR / "leave_one_library_out_metrics.csv", index=False)
    lolo_history.to_csv(RUN_DIR / "leave_one_library_out_history.csv", index=False)
    display(lolo_metrics)
else:
    print("Skipped strict LOLO. Set RUN_STRICT_LOLO=True when needed.")

Skipped strict LOLO. Set RUN_STRICT_LOLO=True when needed.


## Next step

Open `Model_CorePromoter_recursive_design.ipynb`, keep `CORE_MODEL_VARIANT = "tss_pas"`, and run the existing recursive design cells. The recursive notebook loads the frozen `weights_CorePromoter_tss_pas.pt`; it does not retrain the neural network.